# Credit Card Default Prediction — Training Notebook

**Stage 1.5** of the project (see `docs/progress.md`). Run this in Google Colab.

Dataset: [Default of Credit Card Clients](https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset)

Workflow reminder (full detail in `docs/mlflow-workflow.md`):
1. Everything here logs to a **local** MLflow tracking URI (`file:./mlruns`) — no network setup needed from Colab.
2. At the end of the session, zip `mlruns/` and download it, along with the best model's artifacts.
3. Drop `mlruns/` into `infra/mlflow/data/mlruns` on your machine to browse everything in the local MLflow UI.
4. Drop the exported model + preprocessing artifacts into `ml/artifacts/` for the FastAPI backend (Stage 2).

## 0. Setup

In [ ]:
!pip install -q "mlflow==2.16.2" torch scikit-learn imbalanced-learn pandas matplotlib seaborn optuna

# Pinned to 2.16.2 on purpose — matches infra/mlflow/Dockerfile exactly.
#
# Why this matters: `pip install mlflow` unpinned pulls MLflow 3.x, which put the
# FileStore backend (the plain-folder `mlruns/`, no database) into "maintenance mode" —
# `mlflow.set_experiment(...)` raises MlflowException unless you opt back in with
# MLFLOW_ALLOW_FILE_STORE=true. Setting that env var would silence the error, but it
# doesn't guarantee the on-disk format 3.x writes is byte-for-byte what 2.16.2 (running
# in the Docker server) reads back. Pinning both sides to the same version removes that
# risk entirely — this is the fix if you hit:
#   MlflowException: The filesystem tracking backend (e.g., './mlruns') is in
#   maintenance mode ...

import mlflow

# Local file-store tracking — matches the FileStore backend used by the
# Dockerized MLflow server, so mlruns/ can be copied over directly later.
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("credit-card-default")

## 1. Load Data

Download the dataset from Kaggle (via `kagglehub` or manual upload) into `ml/data/` conventions — in Colab, just load it into a DataFrame directly.

In [ ]:
# --- Option A (default): manual upload ---------------------------------------
# Download the CSV once from Kaggle (button on the dataset page), then run this
# cell and pick the file when the widget appears. No Kaggle account/API token
# juggling inside Colab — one less thing to debug on day one.
from google.colab import files

uploaded = files.upload()  # pick the downloaded CSV in the dialog
csv_filename = next(iter(uploaded))  # grabs whatever filename you uploaded

# --- Option B: kagglehub (uncomment if you'd rather not re-upload each session) ---
# import kagglehub
# path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")
# csv_filename = f"{path}/UCI_Credit_Card.csv"

import pandas as pd

df = pd.read_csv(csv_filename)

# Quirk #1: the ID column is a row identifier, not a feature — drop it.
df = df.drop(columns=["ID"])

# Quirk #2 (easy to miss): the repayment-status columns are named
# PAY_0, PAY_2, PAY_3, PAY_4, PAY_5, PAY_6 — there is no PAY_1. That's not a
# typo in this notebook, it's how the original dataset is labeled. PAY_0 is
# the most recent month, PAY_6 the oldest.

print(df.shape)
df.head()

## 2. Exploratory Data Analysis

- Target distribution (`default.payment.next.month`) — quantify the class imbalance
- Univariate distributions: `LIMIT_BAL`, `AGE`, `BILL_AMT1-6`, `PAY_AMT1-6`
- `PAY_0..PAY_6` repayment status patterns vs. default
- Data quality: undocumented codes in `EDUCATION` (0, 5, 6) and `MARRIAGE` (0)
- Correlation heatmap

In [ ]:
# 2.1 Target distribution — how imbalanced are we actually dealing with?
target = "default.payment.next.month"

counts = df[target].value_counts().sort_index()
pct = df[target].value_counts(normalize=True).sort_index() * 100

print("Class counts:\n", counts)
print("\nClass %:\n", pct.round(2))

# The number that matters most for everything downstream: if you did nothing
# clever and just predicted "no default" for every single customer, this is
# the accuracy you'd get for free — with zero predictive power.
naive_accuracy = counts[0] / counts.sum()
print(f"\nNaive 'always predict no-default' accuracy: {naive_accuracy:.2%}")
print("Any model has to beat THIS meaningfully on precision/recall — not just accuracy.")

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=["No Default (0)", "Default (1)"], y=counts.values, ax=ax, palette=["#4C72B0", "#C44E52"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f"{v}\n({pct.values[i]:.1f}%)", ha="center")
ax.set_ylabel("Number of customers")
ax.set_title("Target distribution — default.payment.next.month")
plt.show()

**Implication:** ~78/22 split — not extreme (like fraud's 99.9/0.1), but enough that accuracy is a
misleading headline metric. A model that just memorizes the majority class scores ~78% "accuracy" while
being useless. This is *why* Section 4 (Handling Class Imbalance) and the metrics we track from here on
(precision, recall, F1, PR-AUC) exist — keep this naive-accuracy number around as the floor everything else
has to clear.

In [ ]:
# 2.2 Missing values & duplicates — quick hygiene check before trusting anything else
print("Missing values per column (top 5, should be all 0 for this dataset):")
print(df.isnull().sum().sort_values(ascending=False).head())

print(f"\nExact duplicate rows: {df.duplicated().sum()}")

# Not glamorous, but skipping this step and finding out three sections later
# that you had NaNs silently propagating through a scaler is a worse afternoon.

### 2.3 Data quality: undocumented category codes

The dataset's documentation defines:
- `EDUCATION`: 1=graduate school, 2=university, 3=high school, 4=others
- `MARRIAGE`: 1=married, 2=single, 3=others

But real values in the columns don't stop there — let's look.

In [ ]:
print("EDUCATION value counts:")
print(df["EDUCATION"].value_counts().sort_index())
# -> you'll see 0, 5, 6 show up too — undocumented codes, almost certainly
#    data-entry artifacts or an "unknown" bucket that never made it into the docs.

print("\nMARRIAGE value counts:")
print(df["MARRIAGE"].value_counts().sort_index())
# -> 0 shows up here too, outside the documented 1/2/3.

# Not fixing these here — this is EDA, we're only *finding* the issue. The fix
# (folding 0/5/6 into EDUCATION's "others"=4, and 0 into MARRIAGE's "others"=3)
# happens in Section 3, where every preprocessing decision belongs together.

### 2.4 Numeric distributions — shape matters for a neural net

MLPs are sensitive to feature scale (that's *why* Section 3 ends in `StandardScaler`), and gradient
descent behaves badly when a feature is heavily right-skewed with a long tail of outliers — a few whales
with huge bill amounts can dominate the loss early in training. Let's see which columns actually look like
that before deciding anything.

In [ ]:
# LIMIT_BAL (credit limit) and AGE — the two "plain" continuous features
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["LIMIT_BAL"], bins=40, ax=axes[0], color="#4C72B0")
axes[0].set_title("LIMIT_BAL distribution")
sns.histplot(df["AGE"], bins=40, ax=axes[1], color="#55A868")
axes[1].set_title("AGE distribution")
plt.tight_layout()
plt.show()

print("LIMIT_BAL skew:", df["LIMIT_BAL"].skew().round(2))
print("AGE skew:", df["AGE"].skew().round(2))

# BILL_AMT1 / PAY_AMT1 — representative of the 6 bill/payment columns each.
# Plotting raw vs log1p side-by-side to make the skew concrete rather than
# just quoting a skew number.
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.histplot(df["BILL_AMT1"], bins=50, ax=axes[0, 0], color="#C44E52")
axes[0, 0].set_title(f"BILL_AMT1 (raw) — skew={df['BILL_AMT1'].skew():.2f}")
sns.histplot(np.sign(df["BILL_AMT1"]) * np.log1p(np.abs(df["BILL_AMT1"])), bins=50, ax=axes[0, 1], color="#C44E52")
axes[0, 1].set_title("BILL_AMT1 (signed log1p) — for comparison only")

sns.histplot(df["PAY_AMT1"], bins=50, ax=axes[1, 0], color="#8172B2")
axes[1, 0].set_title(f"PAY_AMT1 (raw) — skew={df['PAY_AMT1'].skew():.2f}")
sns.histplot(np.log1p(df["PAY_AMT1"]), bins=50, ax=axes[1, 1], color="#8172B2")
axes[1, 1].set_title("PAY_AMT1 (log1p) — for comparison only")
plt.tight_layout()
plt.show()

# Note: BILL_AMT can be negative (overpayment/credit balance), which is why the
# signed-log trick is used instead of a plain log1p — plain log1p breaks on
# negative input. This plot is just showing you the shape difference; whether
# we actually apply a log transform is a Section 3 decision, made together
# with the engineered ratio features (a ratio feature can absorb some of this
# skew on its own, which may make an explicit log transform unnecessary).

**Reading these plots:** `LIMIT_BAL` and `AGE` are both right-skewed but mildly — no transform needed,
`StandardScaler` in Section 3 handles this fine. `BILL_AMT1` and `PAY_AMT1` are a different story: the raw
histograms (left column) are a tall spike near zero with a long thin tail stretching far right — most
customers carry small balances/payments, a few carry very large ones. The log1p versions (right column)
pull that tail in and spread the bulk of the mass out, which is closer to the "well-behaved" shape gradient
descent likes. We're not applying the log transform yet — just confirming *why* it's on the table for
Section 3, alongside the ratio features that address the same skew a different way.

### 2.5 Repayment status (`PAY_0..PAY_6`) vs. default — the headline signal

`PAY_0` is last month's repayment status: -1/0 roughly mean "paid on time / revolving credit used
properly", and 1, 2, 3... mean "N months late". If there's one relationship in this dataset that should be
strong, it's this one — let's check, and use it as the sanity check for everything else: if a supposedly
"engineered" feature in Section 3 correlates with default *less* than raw `PAY_0`, that's a signal the
engineering didn't add much.

In [ ]:
pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, pay_cols):
    default_rate_by_status = df.groupby(col)[target].mean().sort_index()
    n_by_status = df.groupby(col).size().sort_index()
    sns.barplot(x=default_rate_by_status.index, y=default_rate_by_status.values, ax=ax, color="#C44E52")
    ax.axhline(naive_accuracy_complement := df[target].mean(), color="gray", linestyle="--", linewidth=1)
    ax.set_title(col)
    ax.set_ylabel("Default rate")
    ax.set_xlabel("Repayment status code")
plt.suptitle("Default rate by repayment status, per month (dashed line = overall default rate)", y=1.02)
plt.tight_layout()
plt.show()

# Reading this: for PAY_0, status <= 0 (paid on time) sits near/below the dashed
# overall-rate line; each step up in "months late" pushes the default rate up,
# often steeply. That monotonic climb is exactly the kind of pattern a linear
# model AND a neural net can both exploit easily — this is your strongest
# individual predictor before any feature engineering happens at all.

**Reading this plot:** for `PAY_0` especially, bars at status ≤ 0 (paid on time / no consumption) sit at or
below the dashed overall-default-rate line, then climb — often sharply — as the status code increases (more
months late). The same shape repeats, a bit weaker, across `PAY_2` through `PAY_6`, and it fades slightly
the further back in time you go (recent behavior predicts next-month default better than 6-month-old
behavior — makes intuitive sense). Concretely: **`PAY_0` is the single most useful raw column in this
dataset**, which is exactly what the ranked correlation list in 2.6 will confirm numerically.

### 2.6 Correlation heatmap — which raw features already "know" the target?

A heatmap won't catch non-linear relationships (a neural net can), but it's a fast way to see which raw
columns already carry signal and which look like noise before you've engineered anything. Worth revisiting
after Section 3 — if an engineered feature doesn't beat its raw ingredients here, question whether it earns
its place in the model.

In [ ]:
corr = df.corr(numeric_only=True)

# Full heatmap is dense (23 features) — also print just the target correlations,
# sorted, since that ranked list is usually more actionable than eyeballing a grid.
target_corr = corr[target].drop(target).sort_values(key=abs, ascending=False)
print("Features ranked by |correlation| with default:")
print(target_corr.round(3))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=ax)
ax.set_title("Correlation matrix — all numeric features")
plt.tight_layout()
plt.show()

# Expect PAY_0..PAY_6 to dominate the top of that ranked list (confirms 2.5),
# LIMIT_BAL to show a modest negative correlation (higher credit limit ~ lower
# default, likely because it's a proxy for creditworthiness the bank already
# assessed), and the six BILL_AMT columns to correlate strongly WITH EACH
# OTHER (multicollinearity — a customer's bill in month 1 is similar to month
# 2) more than with the target individually. That last point is itself useful:
# six highly-correlated raw columns is a candidate for feature engineering
# (e.g. a trend/slope feature) instead of feeding all six in raw.

**Reading this plot:** the ranked list will show `PAY_0..PAY_6` at the top (confirming 2.5 numerically —
`PAY_0` typically lands around |r| ≈ 0.3-0.4, the strongest of any raw column), `LIMIT_BAL` with a modest
*negative* correlation (banks already priced in creditworthiness when they set the limit, so it's a weak
proxy for the same thing the model is trying to predict), and most `BILL_AMT*`/demographic columns clustered
near zero — individually weak. On the heatmap itself, look for the bright block among `BILL_AMT1..6`: they
correlate strongly with **each other** (0.8+), not just the target. That's multicollinearity, and it's the
concrete evidence behind the "6 columns → 1 trend feature" idea from Section 2.4/2.8.

### 2.7 Default rate across demographic slices

Not because these will necessarily be strong predictors (they usually aren't, compared to `PAY_0`), but
because it's worth knowing whether the model's errors will skew across sex/education/marital-status groups
before it's deployed — that's an error-analysis and fairness question worth having eyes on early, not
something to discover after Section 9.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sex_labels = {1: "Male", 2: "Female"}
sns.barplot(x=df["SEX"].map(sex_labels), y=df[target], ax=axes[0], color="#4C72B0", errorbar=None)
axes[0].set_title("Default rate by SEX")
axes[0].set_ylabel("Default rate")

sns.barplot(x=df["EDUCATION"], y=df[target], ax=axes[1], color="#55A868", errorbar=None)
axes[1].set_title("Default rate by EDUCATION (raw codes incl. undocumented)")

sns.barplot(x=df["MARRIAGE"], y=df[target], ax=axes[2], color="#8172B2", errorbar=None)
axes[2].set_title("Default rate by MARRIAGE (raw codes incl. undocumented)")

plt.tight_layout()
plt.show()

# AGE as a binned view — raw age is noisy, bucketing makes any trend visible
age_bins = pd.cut(df["AGE"], bins=[20, 30, 40, 50, 60, 80], labels=["21-30", "31-40", "41-50", "51-60", "60+"])
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=age_bins, y=df[target], ax=ax, color="#C44E52", errorbar=None)
ax.set_title("Default rate by age bracket")
ax.set_ylabel("Default rate")
plt.show()

**Reading these plots:** expect small, noisy differences rather than a dramatic signal — e.g. `EDUCATION`'s
undocumented codes (0, 5, 6) often show wildly different bars simply because they have very few rows behind
them (low n, high variance, not a real effect — another reason to fold them into "others" in Section 3
rather than trust them as their own category). The age-bracket view usually shows default rate ticking up
slightly at both tails (younger, less credit history / older, fixed income) with a dip in the middle
brackets. None of this rivals `PAY_0` in strength, but it's worth remembering when Section 9's error
analysis looks at whether misclassifications cluster in any of these groups.

### 2.8 EDA summary → what it decides for Section 3

| Finding | Decision it drives |
|---|---|
| ~78/22 class split | Track precision/recall/F1/PR-AUC everywhere, not accuracy; revisit imbalance handling in Section 4 |
| `EDUCATION`/`MARRIAGE` have undocumented codes (0, 5, 6 / 0) | Consolidate into the documented "others" bucket before encoding |
| `BILL_AMT*`/`PAY_AMT*` are heavily right-skewed | `StandardScaler` alone may not be enough — consider engineered ratio/trend features that are naturally less skewed than the raw amounts |
| `PAY_0..PAY_6` dominate correlation with target, and climb monotonically with lateness | Strongest raw signal — any engineered feature (delinquency streak, etc.) should be judged against beating this baseline, not replacing it |
| `BILL_AMT1..6` are highly collinear with each other | A trend/slope feature across the 6 months may capture more than 6 raw correlated columns |
| Demographic slices (`SEX`/`EDUCATION`/`MARRIAGE`/`AGE`) show smaller, noisier effects than `PAY_0` | Still worth encoding as features, but keep an eye on them in Section 9's error analysis for skewed error rates |

Nothing gets fixed in this section on purpose — EDA's job is to *decide* what Section 3 needs to do and
*why*, not to do it. Next: Section 3, acting on this table.

In [ ]:
# 3.1 Fix the undocumented category codes found in EDA 2.3
df_fe = df.copy()

# EDUCATION: fold 0, 5, 6 (undocumented) into 4 ("others") — the documented bucket
# that already means "doesn't fit graduate/university/high-school".
df_fe["EDUCATION"] = df_fe["EDUCATION"].replace({0: 4, 5: 4, 6: 4})

# MARRIAGE: fold 0 (undocumented) into 3 ("others").
df_fe["MARRIAGE"] = df_fe["MARRIAGE"].replace({0: 3})

print("EDUCATION after cleanup:", sorted(df_fe["EDUCATION"].unique()))
print("MARRIAGE after cleanup:", sorted(df_fe["MARRIAGE"].unique()))

### 3.2 Feature engineering

Six new features, each chosen to address a specific thing EDA flagged rather than engineered for its own
sake:

| Feature | What it captures | Why (from EDA) |
|---|---|---|
| `PAY_AVG` | Mean repayment status across the 6 months | Smooths single-month noise out of the strongest raw signal (2.5) |
| `PAY_MAX` | Worst (most-late) status in the 6 months | "Ever seriously late" can matter more than "late on average" |
| `DELINQUENCY_STREAK` | Count of months with status > 0 (late) | A repeat-late customer reads differently than one bad month |
| `BILL_TREND` | Slope of a linear fit across `BILL_AMT1..6` | Replaces 6 collinear columns (2.6) with one "balance rising/falling" number |
| `AVG_BILL` | Mean bill amount across 6 months | Also collapses the 6 collinear `BILL_AMT*` columns |
| `UTILIZATION` | `AVG_BILL / LIMIT_BAL` | Classic credit-risk feature: % of credit limit typically used |
| `PAY_TO_BILL_RATIO` | `AVG_PAY_AMT / AVG_BILL` | Are they paying down what they owe, or barely touching it? |

In [ ]:
pay_status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_amt_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

# --- Repayment-status aggregates ---
df_fe["PAY_AVG"] = df_fe[pay_status_cols].mean(axis=1)
df_fe["PAY_MAX"] = df_fe[pay_status_cols].max(axis=1)
df_fe["DELINQUENCY_STREAK"] = (df_fe[pay_status_cols] > 0).sum(axis=1)

# --- Bill trend: slope of a straight-line fit across the 6 months ---
# BILL_AMT1 is the most recent month, BILL_AMT6 the oldest — reverse so the
# fit reads chronologically (oldest -> newest), so a positive slope means
# "balance climbing toward the target month", which is the intuitive
# direction to reason about.
bill_matrix_chronological = df_fe[bill_cols[::-1]].to_numpy()
months = np.arange(6)


def linear_slope(row: np.ndarray) -> float:
    return np.polyfit(months, row, 1)[0]


df_fe["BILL_TREND"] = np.apply_along_axis(linear_slope, 1, bill_matrix_chronological)

# --- Utilization & payment-to-bill ratio ---
df_fe["AVG_BILL"] = df_fe[bill_cols].mean(axis=1)
avg_pay_amt = df_fe[pay_amt_cols].mean(axis=1)

# LIMIT_BAL is never 0 in this dataset, but guard anyway rather than trust that blindly.
df_fe["UTILIZATION"] = (df_fe["AVG_BILL"] / df_fe["LIMIT_BAL"].replace(0, np.nan)).fillna(0)
# Utilization can go negative (overpayment) or spike very high for edge-case rows —
# clip to a sane range so a handful of outliers don't dominate the scaled feature.
df_fe["UTILIZATION"] = df_fe["UTILIZATION"].clip(-1, 3)

# AVG_BILL can be 0 or negative (no/overpaid balance) — guard the ratio the same way.
df_fe["PAY_TO_BILL_RATIO"] = (avg_pay_amt / df_fe["AVG_BILL"].replace(0, np.nan)).fillna(0)
df_fe["PAY_TO_BILL_RATIO"] = df_fe["PAY_TO_BILL_RATIO"].clip(-5, 5)

engineered_cols = ["PAY_AVG", "PAY_MAX", "DELINQUENCY_STREAK", "BILL_TREND", "AVG_BILL", "UTILIZATION", "PAY_TO_BILL_RATIO"]
df_fe[engineered_cols].describe()

### 3.3 Did the engineering actually help? Checking before we trust it

Same test EDA 2.6 promised: rank the new features by |correlation| with the target, and see where they
land next to the raw columns that fed them. If `PAY_AVG`/`PAY_MAX` don't come close to raw `PAY_0`, or
`BILL_TREND`/`UTILIZATION` don't beat the individual raw `BILL_AMT*` columns, that's a real finding too —
not everything engineered earns its keep, and it's better to find that out now than after training.

In [ ]:
compare_cols = ["PAY_0", "PAY_2", "BILL_AMT1", "LIMIT_BAL"] + engineered_cols
corr_compare = df_fe[compare_cols + [target]].corr(numeric_only=True)[target].drop(target)
corr_compare = corr_compare.reindex(corr_compare.abs().sort_values(ascending=False).index)

# Color raw columns differently from engineered ones so the comparison is visual, not just a table.
colors = ["#4C72B0" if c in engineered_cols else "#999999" for c in corr_compare.index]

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=corr_compare.values, y=corr_compare.index, palette=colors, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlation with default")
ax.set_title("Raw (gray) vs. engineered (blue) features — correlation with target")
plt.tight_layout()
plt.show()

print(corr_compare.round(3))

**Reading this plot:** expect `PAY_MAX` and `PAY_AVG` to land right next to raw `PAY_0` — they're built from
the same signal, smoothed, so "close but not dramatically higher" is a *good* result (it means they're
capturing the same real pattern, not noise). `UTILIZATION` and `BILL_TREND` are the ones worth watching
closely: if either clearly beats every individual `BILL_AMT*` column, that's this notebook's concrete
evidence that collapsing 6 collinear columns into 1 engineered feature was worth doing — exactly the
"feature engineering improving the outcome" effect this project is meant to demonstrate. If one of them
*doesn't* beat the raw columns, keep it anyway (it's still a legitimate, differently-shaped signal an MLP
can combine with others) but don't oversell it in the writeup.

### 3.4 Encode categoricals, then split (before scaling — see the note in the next cell)

In [ ]:
# SEX is already binary (1=male, 2=female) — remap to 0/1 rather than one-hot,
# no information gained from a second dummy column for a 2-level variable.
df_fe["SEX"] = df_fe["SEX"].map({1: 0, 2: 1})

# EDUCATION and MARRIAGE are nominal (no inherent order), even though they're
# stored as ints — one-hot them so the model doesn't invent a false ordering
# (e.g. "high school < university" implying a linear relationship that isn't there).
df_encoded = pd.get_dummies(df_fe, columns=["EDUCATION", "MARRIAGE"], drop_first=True, dtype=int)

feature_columns = [c for c in df_encoded.columns if c != target]
print(f"Final feature count: {len(feature_columns)}")
print(feature_columns)

### 3.5 Stratified train / validation / test split, then scale

**Split before scaling, always.** `StandardScaler` learns a mean and standard deviation from whatever data
you fit it on. Fit it on the full dataset and those statistics have quietly seen the validation/test rows —
a subtle form of data leakage that makes validation metrics look slightly better than what the model will
actually see in production. Fitting only on `X_train` and *applying* (`.transform`, never `.fit` again) to
val/test keeps the held-out sets honest.

Split is 70/15/15 (train/val/test), stratified on the target both times so the ~22% default rate from EDA
2.1 is preserved in all three sets — otherwise a random split could hand you an unrepresentative validation
set purely by chance, given how much data is in the minority class.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

X = df_encoded[feature_columns]
y = df_encoded[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print("\nDefault rate per split (should all be ~22%, confirming stratify worked):")
for name, y_split in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  {name}: {y_split.mean():.2%}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit ONLY on train
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# feature_columns (order!) + scaler are both needed, unchanged, at inference time
# in the FastAPI backend — this exact list/order is what gets exported in Section 10.
print(f"\nfeature_columns is now the contract the backend must match: {len(feature_columns)} columns, in this order.")

### 4.0 Shared infrastructure: model + training loop

Comparing imbalance strategies means training a model four different ways — so the reusable architecture
and training loop have to exist *now*, even though Section 5 is where we formally document and finalize the
architecture's design choices. This isn't duplicated later — the class defined below is the same one used,
unchanged, all the way through Section 9.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)

torch.manual_seed(RANDOM_STATE)


class CreditDefaultMLP(nn.Module):
    """Configurable MLP. Every constructor argument here becomes a field in
    model_config.json (Section 10) — the FastAPI backend rebuilds this exact
    class from that file before loading the trained weights, so nothing here
    is allowed to be "just a default I'll remember"."""

    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.2, use_batchnorm=True, init_scheme="he"):
        super().__init__()
        self.config = {
            "input_dim": input_dim,
            "hidden_dims": list(hidden_dims),
            "dropout": dropout,
            "use_batchnorm": use_batchnorm,
            "init_scheme": init_scheme,
        }
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        # Single output logit — BCEWithLogitsLoss applies sigmoid internally,
        # so there's no sigmoid here. Keep it that way; a double-sigmoid bug
        # (one here, one in the loss) is a classic silent-training-failure trap.
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights(init_scheme)

    def _init_weights(self, scheme: str):
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                if scheme == "zero":
                    nn.init.zeros_(m.weight)
                elif scheme == "xavier":
                    nn.init.xavier_normal_(m.weight)
                elif scheme == "he":
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                else:
                    raise ValueError(f"Unknown init_scheme: {scheme}")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _to_tensor(X, y=None):
    X_t = torch.tensor(np.asarray(X), dtype=torch.float32)
    if y is None:
        return X_t
    y_t = torch.tensor(np.asarray(y), dtype=torch.float32)
    return X_t, y_t


def evaluate(model, X, y, threshold=0.5):
    """Metrics that matter for an imbalanced target — accuracy is included
    for reference only, never as the deciding number (EDA 2.1)."""
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(_to_tensor(X))).numpy()
    preds = (probs >= threshold).astype(int)
    y_arr = np.asarray(y)
    return {
        "accuracy": accuracy_score(y_arr, preds),
        "precision": precision_score(y_arr, preds, zero_division=0),
        "recall": recall_score(y_arr, preds, zero_division=0),
        "f1": f1_score(y_arr, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_arr, probs),
        "pr_auc": average_precision_score(y_arr, probs),
    }


_OPTIMIZERS = {
    "sgd": lambda params, lr, wd: torch.optim.SGD(params, lr=lr, weight_decay=wd),
    "momentum": lambda params, lr, wd: torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd),
    "rmsprop": lambda params, lr, wd: torch.optim.RMSprop(params, lr=lr, weight_decay=wd),
    "adam": lambda params, lr, wd: torch.optim.Adam(params, lr=lr, weight_decay=wd),
    "adamw": lambda params, lr, wd: torch.optim.AdamW(params, lr=lr, weight_decay=wd),
}


def train_model(model, X_train, y_train, X_val, y_val, epochs=25, lr=1e-3, batch_size=256,
                 weight_decay=0.0, pos_weight=None, optimizer_name="adam",
                 early_stopping=False, patience=5, verbose=False):
    """Generic training loop reused (unchanged) from here through Section 9.
    Returns per-epoch {train_loss, val_loss} history for plotting."""
    X_t, y_t = _to_tensor(X_train, y_train)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

    pw = torch.tensor(pos_weight, dtype=torch.float32) if pos_weight is not None else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = _OPTIMIZERS[optimizer_name](model.parameters(), lr, weight_decay)

    X_val_t, y_val_t = _to_tensor(X_val, y_val)
    history = {"train_loss": [], "val_loss": []}
    best_val_loss, best_state, epochs_no_improve = float("inf"), None, 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(loader.dataset)

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:3d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f}")

        if early_stopping:
            if val_loss < best_val_loss - 1e-4:
                best_val_loss, epochs_no_improve = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    if verbose:
                        print(f"  early stopping at epoch {epoch} (no val_loss improvement for {patience} epochs)")
                    break

    if early_stopping and best_state is not None:
        model.load_state_dict(best_state)  # roll back to the best checkpoint, not the last one

    return history


print("Shared model class + train_model() + evaluate() ready.")

### 4.1 Four strategies, trained identically except for how they see the imbalance

Same architecture, same 25 epochs, same optimizer, every time — the *only* thing that changes between runs
is how class imbalance is handled. That's what makes the comparison fair; changing the model too would
confound the result.

**Critical rule: resampling only ever touches the training set.** `X_val`/`y_val` stay exactly as EDA found
them (~22% default) for every run below — validating against an artificially rebalanced set would tell you
how good the model is at the *rebalanced* problem, not the real one.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

input_dim = X_train_scaled.shape[1]
n_pos = int(y_train.sum())
n_neg = int(len(y_train) - n_pos)
class_weight_pos_weight = n_neg / n_pos  # BCEWithLogitsLoss pos_weight: how many negatives per positive

X_train_smote, y_train_smote = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train_scaled, y_train)
X_train_under, y_train_under = RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(X_train_scaled, y_train)

print(f"Original train: {len(y_train)} rows, {y_train.mean():.1%} default")
print(f"SMOTE train:     {len(y_train_smote)} rows, {y_train_smote.mean():.1%} default")
print(f"Undersampled:    {len(y_train_under)} rows, {y_train_under.mean():.1%} default")
print(f"class-weighted pos_weight = {class_weight_pos_weight:.2f} (no resampling, loss reweighted instead)")

strategies = {
    "baseline":       dict(X=X_train_scaled, y=y_train,        pos_weight=None),
    "class_weighted": dict(X=X_train_scaled, y=y_train,        pos_weight=class_weight_pos_weight),
    "smote":          dict(X=X_train_smote,  y=y_train_smote,  pos_weight=None),
    "undersampled":   dict(X=X_train_under,  y=y_train_under,  pos_weight=None),
}

imbalance_results = {}
for name, cfg in strategies.items():
    torch.manual_seed(RANDOM_STATE)  # same starting weights across strategies, for a fair comparison
    model = CreditDefaultMLP(input_dim=input_dim)
    with mlflow.start_run(run_name=f"imbalance-{name}"):
        mlflow.log_params({"strategy": name, "epochs": 25, "lr": 1e-3, "optimizer": "adam", "hidden_dims": "64,32"})
        train_model(model, cfg["X"], cfg["y"], X_val_scaled, y_val, epochs=25, pos_weight=cfg["pos_weight"])
        metrics = evaluate(model, X_val_scaled, y_val)  # always the untouched, imbalanced validation set
        mlflow.log_metrics(metrics)
    imbalance_results[name] = metrics
    print(f"{name:16s} -> " + " ".join(f"{k}={v:.3f}" for k, v in metrics.items()))

In [ ]:
results_df = pd.DataFrame(imbalance_results).T
plot_metrics = ["precision", "recall", "f1", "pr_auc"]

fig, ax = plt.subplots(figsize=(10, 5))
results_df[plot_metrics].plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52", "#55A868", "#8172B2"])
ax.axhline(imbalance_results["baseline"]["f1"], color="gray", linestyle="--", linewidth=1, label="baseline F1")
ax.set_ylabel("Score (validation set)")
ax.set_title("Imbalance strategy comparison — precision / recall / F1 / PR-AUC")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

results_df.round(3)

**Reading this plot:** watch the trade-off, not just which bar is tallest. `baseline` typically has decent
precision but weak recall — with no help, the model leans toward predicting the majority class, so it
misses real defaulters. `undersampled` usually swings hard the other way (recall up, precision down) because
throwing away most of the majority class makes the model over-eager to flag default. `class_weighted` and
`smote` tend to land in between, often with the best F1/PR-AUC of the four — `class_weighted` is generally
the cheaper win (no synthetic data, no risk of SMOTE generating unrealistic interpolated customers) and is
a reasonable default going forward. Whichever wins on *your* run, that's the strategy carried into Section 8
(hyperparameter tuning) — write down which one and why before moving on, since "best F1 today" is a claim
you'll want to be able to re-derive later. This comparison used a fixed 25-epoch budget on a fixed
architecture purely to rank strategies fairly against each other; the actual final model gets a full,
properly-tuned training run in Section 8-9.

In [ ]:
# CreditDefaultMLP was already defined in Section 4.0 (it had to be, to run the
# imbalance comparison) — this cell just instantiates the default config and
# inspects it, as the formal "here is the architecture" checkpoint for the rest
# of the notebook. Sections 6-9 reuse this same class with different config values.

default_model = CreditDefaultMLP(input_dim=input_dim)
print(default_model)

n_params = sum(p.numel() for p in default_model.parameters())
n_trainable = sum(p.numel() for p in default_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {n_params:,} (trainable: {n_trainable:,})")
print(f"\nConfig (this is what becomes model_config.json in Section 10):\n{default_model.config}")

**Design decisions, and why:**

- **`hidden_dims=(64, 32)` — two hidden layers, narrowing.** Deep enough to combine features non-linearly
  (the reason to use an MLP over logistic regression at all), narrowing toward the output is the standard
  "funnel" shape; wide/deep options get their own fair test in Section 8, not guessed here.
- **`ReLU` activations.** Cheap, doesn't saturate for positive inputs (avoids the vanishing-gradient failure
  mode `sigmoid`/`tanh` are prone to in deeper nets) — which is also *why* `init_scheme="he"` is the default:
  He initialization is derived specifically for ReLU's variance behavior (Section 6 shows what happens with
  the wrong pairing).
- **`BatchNorm1d` after every `Linear`, before the activation.** Normalizes each layer's input distribution
  during training, which generally lets you train faster/more stably and acts as a mild regularizer —
  ablated explicitly in Section 7.
- **`Dropout` after every activation.** The other regularizer under test in Section 7 — randomly zeroing
  units during training so the network can't over-rely on any single one.
- **Output is a raw logit, no final sigmoid.** `nn.BCEWithLogitsLoss` (used in `train_model`) combines
  sigmoid + binary cross-entropy in one numerically-stable operation. Applying `torch.sigmoid()` is done
  separately at *inference* time (see `evaluate()` in 4.0) — never inside the model itself.

In [ ]:
# Carry forward the Section 4 winner automatically (by validation F1) rather than
# hardcoding an assumption — whichever strategy actually won on your run is the
# one used from here on.
chosen_strategy = results_df["f1"].idxmax()
print(f"Section 4 winner by F1: '{chosen_strategy}' — using it for every experiment from here on.")

chosen_cfg = strategies[chosen_strategy]
X_train_chosen, y_train_chosen = chosen_cfg["X"], chosen_cfg["y"]
pos_weight_chosen = chosen_cfg["pos_weight"]

init_schemes = ["zero", "xavier", "he"]
init_histories = {}
init_results = {}

for scheme in init_schemes:
    torch.manual_seed(RANDOM_STATE)
    model = CreditDefaultMLP(input_dim=input_dim, init_scheme=scheme)
    with mlflow.start_run(run_name=f"init-{scheme}"):
        mlflow.log_params({"init_scheme": scheme, "strategy": chosen_strategy, "epochs": 25})
        history = train_model(model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
                               epochs=25, pos_weight=pos_weight_chosen)
        metrics = evaluate(model, X_val_scaled, y_val)
        mlflow.log_metrics(metrics)
    init_histories[scheme] = history
    init_results[scheme] = metrics
    print(f"{scheme:8s} -> final train_loss={history['train_loss'][-1]:.4f}  " +
          " ".join(f"{k}={v:.3f}" for k, v in metrics.items()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for scheme in init_schemes:
    axes[0].plot(init_histories[scheme]["train_loss"], label=scheme)
axes[0].set_title("Training loss per epoch, by weight init")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss (train)")
axes[0].legend()

init_results_df = pd.DataFrame(init_results).T
init_results_df[["precision", "recall", "f1"]].plot(kind="bar", ax=axes[1], color=["#4C72B0", "#C44E52", "#55A868"])
axes[1].set_title("Final validation metrics, by weight init")
axes[1].set_xticklabels(init_schemes, rotation=0)

plt.tight_layout()
plt.show()

**Reading the left plot — this is the important one:** `zero` init should sit visibly *above* the other two
lines and barely move across epochs. That's not random bad luck — it's structural. If every weight in a
layer starts at exactly 0, every neuron in that layer computes the exact same output and receives the exact
same gradient during backprop, epoch after epoch. They never differentiate from each other, so the whole
layer behaves like a single neuron no matter how wide you make it — this is the "symmetry problem" that
makes zero-init a genuine failure mode, not just a bad idea in theory.

**Right plot:** `xavier` and `he` should both learn fine and land close to each other on precision/recall/F1
— on a network this shallow, the gap between them is usually small. It grows on deeper networks, since
Xavier's variance assumption was derived for `sigmoid`/`tanh` and He's for `ReLU` specifically — the
mismatch compounds layer over layer. Takeaway to remember for the backend: whichever of `xavier`/`he` wins
here, and `zero` staying broken, is exactly why `he` is `CreditDefaultMLP`'s default — this isn't an
arbitrary choice, it's the one this experiment justifies.

In [ ]:
reg_configs = {
    "none":               dict(dropout=0.0, use_batchnorm=False, weight_decay=0.0,  early_stopping=False),
    "dropout_only":       dict(dropout=0.3, use_batchnorm=False, weight_decay=0.0,  early_stopping=False),
    "batchnorm_only":     dict(dropout=0.0, use_batchnorm=True,  weight_decay=0.0,  early_stopping=False),
    "weight_decay_only":  dict(dropout=0.0, use_batchnorm=False, weight_decay=1e-4, early_stopping=False),
    "early_stopping_only":dict(dropout=0.0, use_batchnorm=False, weight_decay=0.0,  early_stopping=True),
    "all_combined":       dict(dropout=0.3, use_batchnorm=True,  weight_decay=1e-4, early_stopping=True),
}

# 40 epochs (up from 25) — long enough that "none" has room to visibly overfit,
# which is the whole point of this ablation.
reg_histories = {}
reg_results = {}

for name, cfg in reg_configs.items():
    torch.manual_seed(RANDOM_STATE)
    model = CreditDefaultMLP(input_dim=input_dim, dropout=cfg["dropout"], use_batchnorm=cfg["use_batchnorm"])
    with mlflow.start_run(run_name=f"reg-{name}"):
        mlflow.log_params({**cfg, "strategy": chosen_strategy, "epochs": 40})
        history = train_model(model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
                               epochs=40, weight_decay=cfg["weight_decay"], pos_weight=pos_weight_chosen,
                               early_stopping=cfg["early_stopping"], patience=5)
        metrics = evaluate(model, X_val_scaled, y_val)
        mlflow.log_metrics(metrics)
    reg_histories[name] = history
    reg_results[name] = metrics
    gap = history["val_loss"][-1] - history["train_loss"][-1]
    print(f"{name:20s} -> epochs_ran={len(history['train_loss']):2d}  train/val gap={gap:+.4f}  f1={metrics['f1']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(reg_histories["none"]["train_loss"], label="train")
axes[0].plot(reg_histories["none"]["val_loss"], label="val")
axes[0].set_title("'none' — no regularization")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")
axes[0].legend()

axes[1].plot(reg_histories["all_combined"]["train_loss"], label="train")
axes[1].plot(reg_histories["all_combined"]["val_loss"], label="val")
axes[1].set_title("'all_combined' — dropout + batchnorm + weight decay + early stop")
axes[1].set_xlabel("Epoch")
axes[1].legend()

reg_results_df = pd.DataFrame(reg_results).T
reg_results_df["f1"].plot(kind="bar", ax=axes[2], color="#4C72B0")
axes[2].set_title("Final validation F1, by configuration")
axes[2].set_xticklabels(reg_results_df.index, rotation=45, ha="right")

plt.tight_layout()
plt.show()

**Reading the left two plots — the "generalization gap":** in `'none'` (left), watch the train-loss line
keep dropping while val-loss flattens out or starts climbing back up — the widening space between the two
lines *is* overfitting, made visible: the model is increasingly memorizing training examples rather than
learning patterns that hold on unseen data. In `'all_combined'` (middle), the two lines should stay much
closer together across all 40 epochs — that's the four regularizers doing their job, each a different
mechanism for the same goal:
- **Dropout** forces redundancy (no single unit can be relied on), so the network can't memorize as easily.
- **Batch norm** stabilizes each layer's input distribution, indirectly discouraging extreme, overfit-prone weights.
- **Weight decay (L2)** directly penalizes large weights in the loss function — simpler decision boundaries generalize better.
- **Early stopping** just refuses to keep training past the point where val_loss stopped improving — cheapest fix of the four.

**Right plot:** don't be surprised if `'none'` still scores a *respectable* F1 — this dataset/architecture
combo isn't large enough to overfit catastrophically in 40 epochs. The story here is the loss-curve shape,
not necessarily a dramatic F1 gap; regularization's real value shows up more as training runs longer or the
model gets larger (exactly what Section 8's hyperparameter search will explore).

In [ ]:
# Shared trial-runner so grid/random/Optuna all score candidates identically —
# same epoch budget, same data, same MLflow logging shape. Only how each method
# *picks* the next candidate differs, which is the whole point of the comparison.
HPO_EPOCHS = 20


def run_trial(hidden_dims, dropout, lr, weight_decay, optimizer_name, batch_size, run_name, log_to_mlflow=True):
    torch.manual_seed(RANDOM_STATE)
    model = CreditDefaultMLP(input_dim=input_dim, hidden_dims=hidden_dims, dropout=dropout)
    params = {
        "hidden_dims": str(hidden_dims), "dropout": dropout, "lr": lr,
        "weight_decay": weight_decay, "optimizer": optimizer_name, "batch_size": batch_size,
    }
    if log_to_mlflow:
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params(params)
            train_model(model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
                        epochs=HPO_EPOCHS, lr=lr, batch_size=batch_size, weight_decay=weight_decay,
                        pos_weight=pos_weight_chosen, optimizer_name=optimizer_name,
                        early_stopping=True, patience=4)
            metrics = evaluate(model, X_val_scaled, y_val)
            mlflow.log_metrics(metrics)
    else:
        train_model(model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
                    epochs=HPO_EPOCHS, lr=lr, batch_size=batch_size, weight_decay=weight_decay,
                    pos_weight=pos_weight_chosen, optimizer_name=optimizer_name,
                    early_stopping=True, patience=4)
        metrics = evaluate(model, X_val_scaled, y_val)
    return {**params, **metrics}


print(f"run_trial() ready — every HPO method below scores candidates over {HPO_EPOCHS} epochs, identically.")

### 8.1 Grid search — exhaustive over a small, deliberately coarse grid

Every combination of a handful of values for a few knobs. Guaranteed to check everywhere in the grid, but
the grid has to stay small — it grows multiplicatively with every dimension you add (3 learning rates × 2
architectures × 2 dropout rates = 12 trials; add one more 3-value dimension and it's 36). That combinatorial
cost is the whole reason random search and Optuna exist — see 8.2/8.3.

In [ ]:
import itertools

grid = {
    "lr": [1e-2, 1e-3, 1e-4],
    "hidden_dims": [(32,), (64, 32)],
    "dropout": [0.2, 0.4],
}
grid_combos = list(itertools.product(grid["lr"], grid["hidden_dims"], grid["dropout"]))
print(f"Grid search: {len(grid_combos)} trials ({' x '.join(str(len(v)) for v in grid.values())})")

grid_trials = []
for i, (lr, hidden_dims, dropout) in enumerate(grid_combos):
    result = run_trial(hidden_dims, dropout, lr, weight_decay=1e-4, optimizer_name="adam",
                        batch_size=256, run_name=f"grid-{i:02d}")
    grid_trials.append(result)
    print(f"  [{i:2d}/{len(grid_combos)}] lr={lr:<7} hidden={str(hidden_dims):<10} dropout={dropout} -> f1={result['f1']:.3f}")

grid_trials_df = pd.DataFrame(grid_trials)
print("\nBest grid trial:")
print(grid_trials_df.loc[grid_trials_df["f1"].idxmax()])

### 8.2 Random search — same trial budget, wider and continuous search space

Same number of trials as the grid (12), but instead of fixed grid points, each trial samples freely from a
*wider* space — including `optimizer` and `batch_size`, which the grid never touched at all. Classic
random-search result (Bergstra & Bengio, 2012): for a fixed budget, sampling randomly across more dimensions
usually beats a grid restricted to few dimensions, because most hyperparameters don't matter equally — grid
search wastes trials being exhaustive along low-impact dimensions.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
N_RANDOM_TRIALS = 12

hidden_dims_choices = [(32,), (64,), (64, 32), (128, 64), (64, 32, 16)]
optimizer_choices = ["sgd", "momentum", "rmsprop", "adam", "adamw"]
batch_size_choices = [64, 128, 256]

random_trials = []
for i in range(N_RANDOM_TRIALS):
    lr = 10 ** rng.uniform(-4, -1.5)  # log-uniform: HP search spaces for lr span orders of magnitude
    hidden_dims = hidden_dims_choices[rng.integers(len(hidden_dims_choices))]
    dropout = rng.uniform(0.1, 0.5)
    weight_decay = 10 ** rng.uniform(-6, -2)
    optimizer_name = optimizer_choices[rng.integers(len(optimizer_choices))]
    batch_size = int(batch_size_choices[rng.integers(len(batch_size_choices))])

    result = run_trial(hidden_dims, dropout, lr, weight_decay, optimizer_name, batch_size, run_name=f"random-{i:02d}")
    random_trials.append(result)
    print(f"  [{i:2d}/{N_RANDOM_TRIALS}] lr={lr:.5f} hidden={str(hidden_dims):<14} dropout={dropout:.2f} "
          f"opt={optimizer_name:<9} batch={batch_size} -> f1={result['f1']:.3f}")

random_trials_df = pd.DataFrame(random_trials)
print("\nBest random trial:")
print(random_trials_df.loc[random_trials_df["f1"].idxmax()])

### 8.3 Optuna — same budget again, but each trial learns from the ones before it

Grid and random search both pick every trial's hyperparameters independently — trial #10 knows nothing
about how trials #1-9 scored. Optuna's default sampler (TPE — Tree-structured Parzen Estimator) builds a
probabilistic model of "which regions of the search space score well" as trials complete, and biases new
suggestions toward those regions (while still exploring). Same 12-trial budget as the other two, so the
comparison in 8.4 is about search *efficiency*, not who got more attempts.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # the per-trial print below is enough noise already
optuna_trials = []


def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 3e-2, log=True)
    hidden_dims_str = trial.suggest_categorical("hidden_dims", [str(h) for h in hidden_dims_choices])
    hidden_dims = eval(hidden_dims_str)  # safe here — hidden_dims_str only ever comes from our own fixed list above
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", optimizer_choices)
    batch_size = trial.suggest_categorical("batch_size", batch_size_choices)

    result = run_trial(hidden_dims, dropout, lr, weight_decay, optimizer_name, batch_size,
                        run_name=f"optuna-{trial.number:02d}")
    optuna_trials.append(result)
    return result["f1"]


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=12, show_progress_bar=False)

optuna_trials_df = pd.DataFrame(optuna_trials)
print(f"\nBest Optuna trial: f1={study.best_value:.3f}")
print(study.best_params)

### 8.4 Which search strategy actually used its budget best?

Same question all three sections above were building toward: for the *same* 12-trial budget, which method
finds a better hyperparameter combination sooner? Plotting "best F1 seen so far" against trial number makes
that concrete instead of just comparing three final numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label, trials_df in [("grid", grid_trials_df), ("random", random_trials_df), ("optuna", optuna_trials_df)]:
    best_so_far = trials_df["f1"].cummax()
    ax.plot(range(1, len(best_so_far) + 1), best_so_far, marker="o", label=label)
ax.set_xlabel("Trial number")
ax.set_ylabel("Best validation F1 seen so far")
ax.set_title("Search efficiency: best-so-far F1 per trial, same 12-trial budget each")
ax.legend()
plt.tight_layout()
plt.show()

# Combine all trials from all three methods and take the single best overall —
# this is the config carried into Section 9's final model.
all_trials_df = pd.concat([
    grid_trials_df.assign(method="grid"),
    random_trials_df.assign(method="random"),
    optuna_trials_df.assign(method="optuna"),
], ignore_index=True)

best_trial = all_trials_df.loc[all_trials_df["f1"].idxmax()]
print(f"Overall best trial (method={best_trial['method']}, f1={best_trial['f1']:.3f}):")
print(best_trial)

**Reading this plot:** `grid`'s line moves in whatever order `itertools.product` happened to generate the
combinations — its "best-so-far" curve is really just an artifact of that fixed order, not a strategy
finding better regions. `random` should climb somewhat unevenly, since each trial is an independent lottery
ticket over a wider space. `optuna` is the one to watch: after its first handful of trials (TPE needs a few
to build its probability model), its curve should climb faster and land at or above the other two — that's
the "learns from prior trials" advantage from 8.3 made visible, not just claimed. If `optuna` doesn't
clearly win on your run, that's a legitimate result too — 12 trials is a small budget for TPE's model to pay
off, and it's worth noting in the writeup rather than expecting the textbook outcome every time. Either way,
`best_trial` above (whichever method it came from) is the single configuration retrained properly — full
epochs, no early cutoff for search-budget reasons — in Section 9.

In [ ]:
# 9.1 Retrain the Section 8 winner properly — full epoch budget, no early cutoff
# for search-speed reasons this time (early stopping still active, but patience
# is generous). X_test/y_test have not been touched by anything above; this is
# their first and only appearance.
final_hidden_dims = eval(best_trial["hidden_dims"]) if isinstance(best_trial["hidden_dims"], str) else best_trial["hidden_dims"]

torch.manual_seed(RANDOM_STATE)
final_model = CreditDefaultMLP(input_dim=input_dim, hidden_dims=final_hidden_dims, dropout=float(best_trial["dropout"]))

with mlflow.start_run(run_name="final-model"):
    final_params = {
        "hidden_dims": str(final_hidden_dims), "dropout": float(best_trial["dropout"]),
        "lr": float(best_trial["lr"]), "weight_decay": float(best_trial["weight_decay"]),
        "optimizer": best_trial["optimizer"], "batch_size": int(best_trial["batch_size"]),
        "strategy": chosen_strategy, "source_search_method": best_trial["method"],
    }
    mlflow.log_params(final_params)
    final_history = train_model(
        final_model, X_train_chosen, y_train_chosen, X_val_scaled, y_val,
        epochs=100, lr=final_params["lr"], batch_size=final_params["batch_size"],
        weight_decay=final_params["weight_decay"], pos_weight=pos_weight_chosen,
        optimizer_name=final_params["optimizer"], early_stopping=True, patience=10, verbose=True,
    )
    test_metrics_default_threshold = evaluate(final_model, X_test_scaled, y_test)
    mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics_default_threshold.items()})
    mlflow.pytorch.log_model(final_model, "model")

print(f"\nTrained {len(final_history['train_loss'])} epochs (early stopping may have cut this short).")
print("Test set metrics @ threshold=0.5:")
for k, v in test_metrics_default_threshold.items():
    print(f"  {k}: {v:.4f}")

### 9.2 Selecting the best run programmatically

Everything above tracked its own `best_trial` in Python, but the actual deliverable requirement is
selecting the winner *from MLflow* — the whole reason every run above was logged with `mlflow.log_metrics`.
`mlflow.search_runs` queries the tracking store directly, which is also how you'd revisit this after
restarting the Colab runtime and losing the Python variables (the logged runs persist in `mlruns/`
regardless).

In [ ]:
all_runs = mlflow.search_runs(experiment_names=["credit-card-default"])

# Every trial across Sections 4, 6, 7, 8 logged an "f1" metric — sort the whole
# experiment by it to confirm the same winner independently of the in-notebook tracking.
ranked = all_runs.sort_values("metrics.f1", ascending=False)[
    ["tags.mlflow.runName", "metrics.f1", "metrics.precision", "metrics.recall", "metrics.pr_auc"]
]
print("Top 10 runs across the entire experiment, by validation F1:")
ranked.head(10)

### 9.3 Confusion matrix, ROC, and Precision-Recall curves — on the test set

Three complementary views of the same predictions. The confusion matrix shows the four raw outcome counts
at one specific threshold (0.5); ROC and PR curves show performance across *every possible* threshold at
once, which matters because 0.5 is an arbitrary default, not a tuned decision — that gets addressed
directly in 9.4.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, precision_recall_curve

final_model.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(final_model(_to_tensor(X_test_scaled))).numpy()
test_preds_default = (test_probs >= 0.5).astype(int)

print(classification_report(y_test, test_preds_default, target_names=["No Default", "Default"]))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

cm = confusion_matrix(y_test, test_preds_default)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred: No Default", "Pred: Default"], yticklabels=["True: No Default", "True: Default"])
axes[0].set_title("Confusion matrix (threshold=0.5)")

fpr, tpr, _ = roc_curve(y_test, test_probs)
axes[1].plot(fpr, tpr, color="#4C72B0", label=f"AUC={test_metrics_default_threshold['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="random guess")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC curve")
axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, test_probs)
baseline_rate = y_test.mean()
axes[2].plot(rec, prec, color="#C44E52", label=f"AP={test_metrics_default_threshold['pr_auc']:.3f}")
axes[2].axhline(baseline_rate, linestyle="--", color="gray", label=f"random guess ({baseline_rate:.2f})")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall curve")
axes[2].legend()

plt.tight_layout()
plt.show()

**Reading the confusion matrix:** the top-right cell (predicted default, actually didn't — false positives)
and bottom-left cell (predicted no-default, actually defaulted — false negatives) are where to focus, not
the diagonal. In a credit-risk setting these two errors usually cost differently — a false negative means
the bank extended credit to someone who then defaulted (direct loss); a false positive means a good
customer got flagged/declined (opportunity cost, friction). Which one your business context weighs more
heavily is exactly what threshold tuning in 9.4 lets you act on.

**Reading ROC vs. PR:** both curves bow toward the top-left/top-right corner the better the model is, and
both beat their "random guess" reference line meaningfully or the model isn't working. **PR is the more
honest curve on this dataset** — ROC's false-positive rate is computed against the large majority class, so
ROC-AUC can look deceptively good on an imbalanced problem (EDA 2.1) even when precision on the minority
class is mediocre. If you only trust one number from this notebook, make it **PR-AUC**, not ROC-AUC.

### 9.4 Threshold tuning — 0.5 was never a decision, it was a default

Sweep the decision threshold and watch precision and recall trade off directly against each other, then
pick the threshold that maximizes F1 (a reasonable balanced default — swap for a business-weighted choice,
e.g. optimizing recall specifically, if false negatives are known to cost more).

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.01)
sweep = []
for t in thresholds:
    preds_t = (test_probs >= t).astype(int)
    sweep.append({
        "threshold": t,
        "precision": precision_score(y_test, preds_t, zero_division=0),
        "recall": recall_score(y_test, preds_t, zero_division=0),
        "f1": f1_score(y_test, preds_t, zero_division=0),
    })
sweep_df = pd.DataFrame(sweep)
best_threshold = float(sweep_df.loc[sweep_df["f1"].idxmax(), "threshold"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sweep_df["threshold"], sweep_df["precision"], label="precision", color="#4C72B0")
ax.plot(sweep_df["threshold"], sweep_df["recall"], label="recall", color="#C44E52")
ax.plot(sweep_df["threshold"], sweep_df["f1"], label="f1", color="#55A868")
ax.axvline(0.5, color="gray", linestyle=":", label="default (0.5)")
ax.axvline(best_threshold, color="black", linestyle="--", label=f"chosen ({best_threshold:.2f})")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / recall / F1 vs. decision threshold (test set)")
ax.legend()
plt.tight_layout()
plt.show()

final_test_metrics = evaluate(final_model, X_test_scaled, y_test, threshold=best_threshold)
print(f"Chosen threshold: {best_threshold:.2f} (was 0.5)")
print("Test metrics @ chosen threshold:")
for k, v in final_test_metrics.items():
    print(f"  {k}: {v:.4f}  (was {test_metrics_default_threshold[k]:.4f} @ 0.5)")

**Reading this plot:** precision (blue) and recall (red) cross somewhere in the middle — below the crossing
point, recall is high and precision is low (catch most defaulters, but with lots of false alarms); above it,
the reverse. The green F1 line peaks near that crossing, which is why "maximize F1" and "pick the
threshold near where precision and recall cross" usually agree. The vertical dashed line vs. dotted line is
the concrete "before/after" from tuning: whatever the gap between them, that's real performance left on the
table by shipping the untuned 0.5 default — this is often a bigger, cheaper win than another round of
architecture search. This `best_threshold` value is exported alongside the model in Section 10 — the
backend has to apply it, not the trained-in-Pytorch default of 0.5.

### 9.5 Error analysis — do the mistakes have a pattern?

Using the chosen threshold's predictions. The question isn't "how many errors" (that's just the confusion
matrix again) but "do false negatives look different from true positives in a way that suggests a fixable
gap" — e.g. if false negatives cluster around a borderline `PAY_0` value, that's a specific, actionable
observation, not just "the model is imperfect."

In [ ]:
test_preds_final = (test_probs >= best_threshold).astype(int)

error_df = X_test.copy()  # unscaled, human-readable values — easier to eyeball than the scaled array
error_df["y_true"] = y_test.values
error_df["y_pred"] = test_preds_final
error_df["p_default"] = test_probs

false_negatives = error_df[(error_df.y_true == 1) & (error_df.y_pred == 0)]
false_positives = error_df[(error_df.y_true == 0) & (error_df.y_pred == 1)]
true_positives = error_df[(error_df.y_true == 1) & (error_df.y_pred == 1)]

print(f"False negatives: {len(false_negatives)}  |  False positives: {len(false_positives)}  |  True positives: {len(true_positives)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.kdeplot(true_positives["PAY_0"], label="True Positive (correctly caught)", ax=axes[0], color="#55A868", fill=True, alpha=0.3)
sns.kdeplot(false_negatives["PAY_0"], label="False Negative (missed)", ax=axes[0], color="#C44E52", fill=True, alpha=0.3)
axes[0].set_title("PAY_0 distribution: caught vs. missed defaulters")
axes[0].legend()

sns.histplot(false_negatives["p_default"], bins=20, ax=axes[1], color="#C44E52")
axes[1].axvline(best_threshold, color="black", linestyle="--", label=f"threshold={best_threshold:.2f}")
axes[1].set_title("Predicted probability for false negatives\n(how close were the misses?)")
axes[1].legend()

plt.tight_layout()
plt.show()

**Reading the left plot:** if the red (missed) curve sits noticeably to the *left* of the green (caught)
curve — i.e., false negatives tend to have lower/better `PAY_0` values than true positives — that's a
specific, explainable failure mode: **the model is missing defaulters who looked fine on recent repayment
history but defaulted anyway**, exactly the customers `PAY_0` alone can't warn you about. That's a concrete
argument for the engineered features from Section 3 (utilization, bill trend) mattering precisely for this
subgroup, and a legitimate direction for future feature work if you continue this project.

**Reading the right plot:** if false negatives cluster close to `best_threshold` rather than near 0, most
misses were "close calls" the model was genuinely uncertain about — a threshold or calibration issue more
than an information gap. If instead there's a cluster of false negatives with very low predicted
probability (confidently, incorrectly, "no default"), that's the more concerning pattern — the model isn't
just miscalibrated there, it's missing signal entirely for that subgroup, which more features or more data
would address better than threshold tuning ever could.

In [ ]:
import json
import pickle
import os

os.makedirs("export/model", exist_ok=True)
os.makedirs("export/preprocessing", exist_ok=True)
os.makedirs("export/metrics", exist_ok=True)

# --- model/model.pt: trained weights only (not the whole object — state_dict
#     is the portable, version-independent way to persist a PyTorch model) ---
torch.save(final_model.state_dict(), "export/model/model.pt")

# --- model/model_config.json: everything needed to reconstruct CreditDefaultMLP
#     and apply its predictions correctly, without needing this notebook again ---
model_config = {
    **final_model.config,
    "decision_threshold": best_threshold,
    "feature_columns": feature_columns,  # exact order the model expects at inference
    "imbalance_strategy": chosen_strategy,
    "training_hyperparams": {
        "lr": final_params["lr"], "weight_decay": final_params["weight_decay"],
        "optimizer": final_params["optimizer"], "batch_size": final_params["batch_size"],
    },
}
with open("export/model/model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

# --- preprocessing/scaler.pkl + feature_columns.json ---
with open("export/preprocessing/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("export/preprocessing/feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2)

# --- metrics/evaluation_report.json — for the README/docs, not needed at inference ---
evaluation_report = {
    "test_metrics_at_threshold_0.5": test_metrics_default_threshold,
    "test_metrics_at_chosen_threshold": final_test_metrics,
    "chosen_threshold": best_threshold,
    "naive_baseline_accuracy": float(naive_accuracy),  # from EDA 2.1 — the floor this model clears
    "imbalance_strategy": chosen_strategy,
    "hyperparameter_search_winner_method": best_trial["method"],
}
with open("export/metrics/evaluation_report.json", "w") as f:
    json.dump(evaluation_report, f, indent=2)

print("Exported to ./export/ :")
for root, _, filenames in os.walk("export"):
    for fn in filenames:
        print(" ", os.path.join(root, fn))

In [ ]:
# Zip both the export/ folder and mlruns/ for download in one go.
!zip -r export.zip export/ -x "*.DS_Store"
!zip -r mlruns_export.zip mlruns/ -x "*.DS_Store"

from google.colab import files

files.download("export.zip")
files.download("mlruns_export.zip")

### 10.1 After downloading — where things go locally

1. Unzip `export.zip` and copy its contents into `ml/artifacts/` in the repo, matching the same
   `model/`, `preprocessing/`, `metrics/` layout — these become the FastAPI backend's inputs in Stage 2.
2. Follow [mlflow-workflow.md](../../docs/mlflow-workflow.md) to merge `mlruns_export.zip` into
   `infra/mlflow/data/mlruns/`, then `make mlflow-up` + `make mlflow-ui` to browse the full experiment
   history — every run from Sections 4, 6, 7, 8, and this final one — locally.
3. Stage 1.5 is done once both of those are in place. Update `docs/progress.md`'s checklist, or just say so
   in chat — Stage 2 (FastAPI) picks up from here.